# 5. Foundation-model embeddings and the MedGemma prompt

Reproduces **Section 3.4** (how the two representations are built) and **Section 4.6** (how much
the prompt matters):

| output | paper |
|---|---|
| four prompt formulations, standalone AUC | `tab:prompts` |
| the answer distribution saturating and then flipping | Section 3.4 and Section 4.6 |
| BiomedCLIP on the T1ce crop and on the enhancement map | Section 4.5, 0.830 and 0.869 |

The table cells run from `data_cache/`. The generation cells need the BraTS copy plus the
MedGemma and BiomedCLIP weights, and are guarded.

## What each encoder is shown

Both read one axial slice per case, the one with the largest **predicted** tumor area, so the
choice never touches a reference mask.

MedGemma gets two panels side by side: the T1ce crop and the enhancement map of the same region.
Both are windowed in standard deviations of healthy brain, not by each crop's own percentiles.
That detail matters more than it looks: per-crop normalization stretches a non-enhancing tumor's
noise across the full range, so a low-grade and a high-grade case come out equally bright and the
enhancement panel stops carrying the signal it exists for.

BiomedCLIP gets a single crop of the same region and its image embedding is taken directly.

Both embeddings are reduced with PCA to sixteen components, fit inside the training folds only as
part of the cross-validation pipeline. The volumetric features are standardized but not reduced.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

from src import config
print(config.describe())

In [ ]:
from src import cohort, encoders, tables

print("MedGemma:", config.MEDGEMMA_ID)
print("BiomedCLIP:", config.BIOMEDCLIP_ID)
print("panels:", f"{encoders.PANEL_SIZE} x {encoders.PANEL_SIZE} each, side by side")
print("T1ce window, in standard deviations of healthy brain:", encoders.T1CE_WINDOW)
print("enhancement window:", encoders.ENHANCEMENT_WINDOW)

## The four prompts

All four ask for six fields about the tumor. They differ in what surrounds that request.

- **Baseline (categorical)** asks for the six fields and nothing else.
- **Numeric (percentages)** replaces the categories with integers from 0 to 100, on the reasoning
  that a category can saturate at one value and a percentage cannot.
- **Few-shot** adds a written calibration block: what avid, mild and absent enhancement look like
  on the enhancement panel, and a statement that many gliomas do not enhance, that "none" is a
  common and expected answer, and that the tumor should be judged from what is visible rather than
  assumed to be high-grade. There are no image exemplars, so no case label can leak into the
  features and the cohort stays at 369.
- **Few-shot + reasoning** keeps that calibration and asks for a short reasoning step before the
  answer. Only the tokens after the marker are pooled, so the vector represents the conclusion and
  not the reasoning.

In [ ]:
for key, prompt in encoders.PROMPTS.items():
    print("=" * 78)
    print(f"{prompt['label']}   [{key}]")
    print("=" * 78)
    print(prompt["system"])
    print(f"\nuser turn: {prompt['user']}")
    print(f"pooled from: {'after ' + prompt['marker'] if prompt['marker'] else 'the whole answer'}\n")

## What the model actually answers

The baseline prompt saturates: "avid" for all 369 cases, and high-grade for 364 of them, against
76 true low-grade cases. The calibration does break the saturation, and in fact overshoots it in
the other direction: "none" for 298 of 369.

That is worth being precise about, because it is not what makes the few-shot embedding useful. The
literal answer stays a coarse grader either way. What the calibration changes is the perception
encoded in the hidden state, and that is what the table below measures.

In [ ]:
rows = {}
for key in encoders.PROMPTS:
    path = config.MEDGEMMA_ANSWERS[key]
    if path.exists():
        rows[encoders.PROMPTS[key]["label"]] = encoders.answer_distribution(path, "Enhancement")
display(pd.DataFrame(rows).fillna(0).astype(int))

In [ ]:
baseline_answers = encoders.parse_answers(config.MEDGEMMA_ANSWERS["baseline"])
display(baseline_answers["Assessment"].value_counts().to_frame("cases"))

## Generating the embeddings

Guarded, because this is the expensive path: roughly 7 seconds per case for the categorical
prompts and about 12 for the reasoning one, plus the model download. The caches these produce ship
with the repository.

In [ ]:
RUN_INFERENCE = False   # needs the BraTS copy and the model weights

if RUN_INFERENCE:
    for key in encoders.PROMPTS:
        encoders.run_medgemma(key)
    for which in encoders.BIOMEDCLIP_INPUTS:
        encoders.embed_biomedclip_all(which)

## tab:prompts

Each prompt's description embedding scored on its own, under the pooled out-of-fold protocol, the
same one `tab:errcorr` uses. Expected:

| Prompt | Standalone AUC | $\Delta$ vs. baseline | $\Delta$ vs. Vol. |
|---|---|---|---|
| Baseline (categorical) | 0.748 | --- | -0.182 |
| Numeric (percentages) | 0.760 | +0.012 | -0.170 |
| Few-shot | **0.819** | +0.071 | -0.111 |
| Few-shot + reasoning | 0.682 | -0.066 | -0.248 |

The reading: the calibration lifts the embedding the most, by 0.071 over the baseline prompt.
Asking for numeric percentages barely moves it, because the estimated values carry no grade signal
of their own. Adding a reasoning step **hurts**, which is consistent with the failure being
perceptual rather than logical: reasoning talks the model back towards its prior.

Even the strongest prompt stays well below the volumetric baseline of 0.930 on its own, so the
prompt matters only through the fusion in notebook 6, never as a grader in itself.

Note on protocol, since it changes the third decimal: these are pooled out-of-fold AUCs, computed
once over the concatenated fold predictions. Averaging the five per-fold AUCs instead gives
0.752 / 0.761 / 0.818 / 0.686. The caption states which one it is, and this is it.

In [ ]:
blocks, y, cases = cohort.load_blocks()
cohort.inject_encoder_blocks(blocks, cases, keys=(
    "medgemma_fewshot", "medgemma_numeric", "medgemma_fewshot_cot", "biomedclip_enh"))

In [ ]:
prompts = tables.table_prompts(blocks, y)
display(prompts)

published = {"Baseline (categorical)": 0.748, "Numeric (percentages)": 0.760,
             "Few-shot": 0.819, "Few-shot $+$ reasoning": 0.682}
display(tables.compare_to_paper(prompts, "Standalone AUC", published))

In [ ]:
from src.evaluation import block_auc, out_of_fold
from sklearn.metrics import roc_auc_score

protocols = pd.DataFrame([
    {"Prompt": label,
     "pooled out-of-fold": round(roc_auc_score(y, out_of_fold(blocks, [key], y)), 4),
     "mean of the five folds": round(block_auc(blocks, [key], y), 4)}
    for label, key in tables.PROMPTS]).set_index("Prompt")
display(protocols)

## The two BiomedCLIP inputs

The slice is fixed and only the image changes, so this isolates what the encoder is shown. The
T1ce crop gives **0.830**; the enhancement map, which is the same signal MedGemma reads, gives
**0.869**.

That the better input scores higher on its own and yet adds less in fusion is the point notebook 6
lands on: the enhancement map is what the volumetric features already measure, so a stronger image
encoder here becomes more redundant, not more complementary.

In [ ]:
inputs = pd.DataFrame([
    {"BiomedCLIP input": "T1ce crop",
     "Standalone AUC": round(roc_auc_score(y, out_of_fold(blocks, ["biomedclip_t1ce"], y)), 3)},
    {"BiomedCLIP input": "enhancement map (T1ce - T1)",
     "Standalone AUC": round(roc_auc_score(y, out_of_fold(blocks, ["biomedclip_enh"], y)), 3)},
]).set_index("BiomedCLIP input")
display(inputs)

Both are above the 0.65 threshold fixed in advance as the level below which the experiment would
have been abandoned, and both stay below the volumetric baseline of 0.930.